# 002 · The match Statement

> **Syntax › Control Flow** · Python 3.14

Python 3.10 added **structural pattern matching**: `match` / `case`. It looks like C#'s `switch`, but it is much closer to C#'s *pattern matching* (`is`, switch expressions, property and list patterns) — it can destructure sequences, dictionaries and objects while it matches.

**In this notebook you will learn:**

- literal, OR (`|`), wildcard (`_`) and capture patterns
- guards (`case ... if ...`) — C#'s `when`
- sequence, mapping and class patterns, and binding sub-patterns with `as`
- the big gotcha: a bare name in a `case` is a **capture**, not a constant
- why there is no fall-through, and why `match` is a statement, not an expression

## C# mental model

| C# | Python |
|---|---|
| `switch (x) { case 1: ... break; }` | `match x:` / `case 1:` (no `break`) |
| `case 1: case 2:` / `1 or 2` | `case 1 \| 2:` |
| `default:` / `_ =>` | `case _:` |
| `case int n when n > 0:` | `case int(n) if n > 0:` |
| list pattern `[var first, ..]` | `case [first, *_]:` |
| property pattern `{ X: 0, Y: var y }` | `case Point(x=0, y=y):` |
| switch **expression** `x switch { ... }` | none — `match` is a statement |

## Literal patterns

The simplest use is the classic `switch` on constant values. Cases are tried top to bottom and the **first** match wins.

```csharp
// C#
switch (status)
{
    case 200: Console.WriteLine("OK"); break;
    case 404: Console.WriteLine("Not Found"); break;
    default:  Console.WriteLine("Unknown"); break;
}
```

In [ ]:
def http_status(status: int) -> str:
    match status:
        case 200:
            return "OK"
        case 404:
            return "Not Found"
        case 500:
            return "Server Error"
        case _:                       # wildcard: matches anything (C#: default)
            return "Unknown"

for code in (200, 404, 418):
    print(code, http_status(code))

## OR patterns and the wildcard

Combine alternatives with `|` (C# 9: `or`). `_` is the **wildcard** — it matches anything and binds nothing. A `case _:` is optional; if no case matches, nothing happens (no exception).

In [ ]:
def day_kind(day: str) -> str:
    match day.lower():
        case "sat" | "sun":
            return "weekend"
        case "mon" | "tue" | "wed" | "thu" | "fri":
            return "weekday"
        case _:
            return "not a day"

print(day_kind("Sun"), day_kind("wed"), day_kind("xyz"))

## No fall-through

C# forbids implicit fall-through and forces you to write `break`. Python simply has **no** fall-through and no `break` inside `match`: exactly one case body runs (or none).

In [ ]:
match 1:
    case 1:
        print("one")                  # runs, then the match is finished
    case 1:
        print("never reached")        # the first matching case wins

## Capture patterns

A **bare name** in a pattern always matches and **binds** the subject to that name — like `var x` in a C# pattern.

```csharp
// C#
var msg = value switch
{
    0 => "zero",
    var other => $"something else: {other}",
};
```

In [ ]:
def describe(value: int) -> str:
    match value:
        case 0:
            return "zero"
        case other:                   # always matches, binds value to 'other'
            return f"something else: {other}"

print(describe(0), "|", describe(7))

## ⚠️ Gotcha: bare names are capture patterns, not constants

This is the single most common `match` bug. You might expect `case RED:` to compare against the constant `RED` — it doesn't. It **captures** into a new variable called `RED`, matches everything, and overwrites your constant!

Python actually refuses to compile a `match` where an irrefutable capture pattern makes later cases unreachable, so the bug shows up as a `SyntaxError`:

In [ ]:
RED = "red"

code = '''
match colour:
    case RED:          # intended: compare to RED. Actual: capture into RED
        print("stop")
    case _:
        print("go")
'''
try:
    compile(code, "<demo>", "exec")
except SyntaxError as e:
    print(f"{type(e).__name__}: {e}")

If the capture is the *last* case, there's no error — the code silently does the wrong thing:

In [ ]:
RED = "red"
colour = "green"

match colour:
    case "blue":
        print("blue")
    case RED:                         # captures! matches "green"
        print("matched RED?!")

print("RED is now:", RED)             # the 'constant' was overwritten

**Fix:** any name containing a dot is treated as a **value pattern** (compared with `==`). Use an `Enum` *(see Classes & OOP › 009 · Enums)* or a class/module attribute:

In [ ]:
from enum import Enum

class Colour(Enum):
    RED = "red"
    GREEN = "green"

def action(c: Colour) -> str:
    match c:
        case Colour.RED:              # dotted name -> value pattern (== comparison)
            return "stop"
        case Colour.GREEN:
            return "go"
        case _:
            return "unknown"

print(action(Colour.RED), action(Colour.GREEN))

## Guards (`if`)

A guard adds an arbitrary condition to a case — C#'s `when` clause. The guard is only checked if the pattern matched, and names captured by the pattern are available in it.

```csharp
// C#
var sign = n switch
{
    0 => "zero",
    var x when x < 0 => "negative",
    _ => "positive",
};
```

In [ ]:
def sign(n: int) -> str:
    match n:
        case 0:
            return "zero"
        case x if x < 0:              # capture, then guard
            return "negative"
        case _:
            return "positive"

print(sign(-5), sign(0), sign(3))

## Sequence patterns

Square brackets (or parentheses) match **sequences** — lists, tuples, `range` etc. — by length and content. `*rest` captures the remaining items, like C# 11 list patterns with `..`.

```csharp
// C#
var text = command switch
{
    [] => "empty",
    ["go", var dir] => $"going {dir}",
    ["drop", .. var items] => $"dropping {items.Length} items",
    _ => "unknown",
};
```

In [ ]:
def run(command: list[str]) -> str:
    match command:
        case []:
            return "empty command"
        case ["quit"]:
            return "bye"
        case ["go", direction]:
            return f"going {direction}"
        case ["drop", *items]:
            return f"dropping {len(items)} item(s): {items}"
        case _:
            return f"unknown: {command}"

for cmd in ([], ["quit"], ["go", "north"], ["drop", "key", "lamp"], ["dance", "now"]):
    print(run(cmd))

A common idiom is matching on `text.split()`. Note that `str` is deliberately **not** treated as a sequence by `match` (otherwise `"go"` would match `[a, b]`).

In [ ]:
match "go north".split():
    case ["go", direction]:
        print("direction:", direction)

match "ab":
    case [a, b]:
        print("matched as sequence")
    case _:
        print("strings are not matched as sequences")

## Mapping patterns

Curly braces match **dicts** (any mapping). Only the listed keys must be present — extra keys are ignored. `**rest` captures the remaining entries. This is great for JSON-like data.

In [ ]:
def handle(event: dict) -> str:
    match event:
        case {"type": "click", "x": x, "y": y}:
            return f"click at ({x}, {y})"
        case {"type": "key", "key": "Enter" | "Return"}:
            return "submit"
        case {"type": "key", "key": key, **rest}:
            return f"key {key!r}, extra: {rest}"
        case {"type": other}:
            return f"unhandled event type {other!r}"
        case _:
            return "not an event"

print(handle({"type": "click", "x": 10, "y": 20, "button": "left"}))
print(handle({"type": "key", "key": "Enter"}))
print(handle({"type": "key", "key": "a", "shift": True}))
print(handle({"type": "scroll"}))
print(handle({}))

## Class patterns

`ClassName(...)` checks the type with `isinstance` and then matches attributes — the equivalent of C#'s type + property patterns. With keyword arguments you name the attributes; dataclasses *(see Classes & OOP › 008 · Dataclasses)* also support **positional** sub-patterns because they generate `__match_args__` automatically.

```csharp
// C#
string Where(object shape) => shape switch
{
    Point { X: 0, Y: 0 } => "origin",
    Point { X: 0, Y: var y } => $"on Y axis at {y}",
    Point(var x, var y) => $"at {x},{y}",   // positional pattern (needs Deconstruct)
    Circle { Radius: > 10 } => "big circle",
    _ => "unknown",
};
```

In [ ]:
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

@dataclass
class Circle:
    centre: Point
    radius: float

def where(shape: object) -> str:
    match shape:
        case Point(x=0, y=0):
            return "origin"
        case Point(x=0, y=y):                      # keyword sub-patterns
            return f"on the Y axis at {y}"
        case Point(x, y):                          # positional (uses __match_args__)
            return f"point at ({x}, {y})"
        case Circle(Point(0, 0), r):               # nested patterns
            return f"circle of radius {r} at the origin"
        case Circle(radius=r) if r > 10:
            return "big circle"
        case _:
            return "unknown"

for s in (Point(0, 0), Point(0, 5), Point(3, 4), Circle(Point(0, 0), 2), Circle(Point(1, 1), 50), "?"):
    print(where(s))

For built-in types, `int(n)`, `str(s)`, `float(f)` etc. match by type and bind the whole value — C#'s `case int n:` type pattern:

In [ ]:
def kind(value: object) -> str:
    match value:
        case bool(b):                 # check bool before int: bool is a subclass of int
            return f"bool {b}"
        case int(n) | float(n):
            return f"number {n}"
        case str(s):
            return f"string {s!r}"
        case None:
            return "None"
        case _:
            return f"other: {type(value).__name__}"

for v in (True, 42, 3.14, "hi", None, [1]):
    print(kind(v))

## Binding with `as`

`pattern as name` matches the sub-pattern **and** binds what it matched — useful with OR patterns, where you want to know which alternative hit. C# does the same with `and var`: `[ "go", ("north" or "south") and var direction ]`.

In [ ]:
def move(command: list[str]) -> str:
    match command:
        case ["go", ("north" | "south" | "east" | "west") as direction]:
            return f"moving {direction}"
        case ["go", *_] as whole:
            return f"invalid: {' '.join(whole)}"
        case _:
            return "?"

print(move(["go", "east"]))
print(move(["go", "up", "fast"]))

## `match` is a statement, not an expression

C# 8 switch **expressions** produce a value: `var x = y switch { ... };`. Python's `match` is a **statement** — it can't appear on the right-hand side of `=`. Options:

1. assign inside each case, or `return` from a function (as above);
2. for simple **value → value** lookups, use a **dict** (often clearer and faster);
3. for **value → behaviour**, use a dict of functions ("dispatch table").

```csharp
// C#
var price = size switch { "S" => 2.0, "M" => 2.5, "L" => 3.0, _ => 0.0 };
```

In [ ]:
size = "M"

# 1. assign in each case
match size:
    case "S": price = 2.0
    case "M": price = 2.5
    case "L": price = 3.0
    case _:   price = 0.0
print(price)

# 2. dict lookup with a default — the idiomatic 'switch expression'
PRICES = {"S": 2.0, "M": 2.5, "L": 3.0}
print(PRICES.get(size, 0.0))

In [ ]:
# 3. dict dispatch: map keys to functions
def add(a, b): return a + b
def sub(a, b): return a - b

OPERATIONS = {
    "+": add,
    "-": sub,
    "*": lambda a, b: a * b,
}

for op in ("+", "-", "*", "/"):
    func = OPERATIONS.get(op)
    print(op, func(6, 3) if func else "unsupported")

## When to use `match`

- Good fit: destructuring structured data: command tokens, JSON-ish dicts, ASTs, dataclass hierarchies
- Good fit: type-based dispatch with nested conditions
- Poor fit: a plain value → value mapping — a `dict` is simpler
- Poor fit: two or three simple conditions — `if`/`elif` is clearer

## Summary

| Concept | C# | Python |
|---|---|---|
| Constant case | `case 1:` / `1 =>` | `case 1:` |
| Alternatives | `1 or 2` | `case 1 \| 2:` |
| Default | `default:` / `_ =>` | `case _:` |
| Capture | `var x` | `case x:` |
| Named constant | `case Colour.Red:` | `case Colour.RED:` (must be dotted!) |
| Guard | `when cond` | `if cond` |
| Type pattern | `case int n:` | `case int(n):` |
| List pattern | `[var a, .. var rest]` | `case [a, *rest]:` |
| Property pattern | `Point { X: 0, Y: var y }` | `case Point(x=0, y=y):` |
| Dictionary match | — | `case {"key": value}:` |
| Bind sub-pattern | `pattern and var x` | `pattern as x` |
| Fall-through | not allowed (needs `break`) | doesn't exist (no `break`) |
| Switch expression | `x switch { ... }` | none — use a `dict` or `return` in cases |

## Exercises

1. Write `fizzbuzz(n)` using `match` on the tuple `(n % 3, n % 5)`.
2. Write `area(shape)` using class patterns for dataclasses `Rect(w, h)` and `Circle(r)`; return `None` for anything else.
3. Given events like `{"action": "deposit", "amount": 50}` and `{"action": "withdraw", "amount": 20}`, write `apply(balance, event)` with mapping patterns and a guard that rejects non-positive amounts.
4. Why does `case MAX_SIZE:` not compare against a module-level `MAX_SIZE = 10`? How do you fix it?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
def fizzbuzz(n: int) -> str:
    match (n % 3, n % 5):
        case (0, 0): return "FizzBuzz"
        case (0, _): return "Fizz"
        case (_, 0): return "Buzz"
        case _:      return str(n)

# 2
import math
from dataclasses import dataclass

@dataclass
class Rect:
    w: float
    h: float

@dataclass
class Circle:
    r: float

def area(shape) -> float | None:
    match shape:
        case Rect(w, h):
            return w * h
        case Circle(r):
            return math.pi * r ** 2
        case _:
            return None

# 3
def apply(balance: float, event: dict) -> float:
    match event:
        case {"amount": amount} if amount <= 0:
            raise ValueError("amount must be positive")
        case {"action": "deposit", "amount": amount}:
            return balance + amount
        case {"action": "withdraw", "amount": amount}:
            return balance - amount
        case _:
            raise ValueError(f"unknown event: {event}")

# 4 — a bare name is a capture pattern: it matches anything and binds
#     the subject to a new MAX_SIZE. Use a dotted name instead
#     (e.g. an Enum member or Config.MAX_SIZE), or a guard: case n if n == MAX_SIZE:
```

</details>